# Video Deepfake Detector v2
**Base**: google/vit-base-patch16-224 + LoRA frame-level | **Output**: saghi776/aiscern-video-detector
**Target**: 80%+ accuracy | **Platform**: Kaggle T4 x2 | ~5h

Strategy: extract N frames per video, run ViT on each, aggregate with IQR-aware pooling.
Datasets: FaceForensics++, Celeb-DF, DeepFake-Vs-Real-Faces, DFDC subset.
Wire into hf-analyze.ts MODELS after push. Labels: {0:'real', 1:'fake'}

In [ ]:
!pip install -q transformers==4.41.0 datasets peft==0.11.0 accelerate==0.30.0 evaluate scikit-learn huggingface_hub opencv-python-headless Pillow
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
print('packages ok')

In [ ]:
import os, torch, random, numpy as np
from PIL import Image

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN','')

BASE_MODEL   = 'google/vit-base-patch16-224'
PUSH_REPO    = 'saghi776/aiscern-video-detector'
CKPT_DIR     = '/kaggle/working/video-ckpt'
IMG_SIZE     = 224
FRAMES_PER_VIDEO = 8     # sample 8 frames evenly per video
BATCH        = 16
GRAD_ACC     = 4
EPOCHS       = 5
LR           = 8e-5
LORA_R       = 16
N_PER_CLASS  = 40000     # frame-level samples
SEED         = 42
random.seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Load frame-level datasets
# For video, we use face image datasets + video frame extracts
from datasets import load_dataset
from torchvision import transforms

real_frames, fake_frames = [], []

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

def proc_img(img):
    if not isinstance(img, Image.Image):
        try: img = Image.fromarray(img)
        except: return None
    return transform(img.convert('RGB'))

# 1. DeepFake-Vs-Real-Faces (face frames from deepfake videos)
print('Loading DeepFake-Vs-Real-Faces...')
try:
    df = load_dataset('arnabdhar/DeepFake-Vs-Real-Faces', split='train', trust_remote_code=True)
    for row in df.select(range(min(len(df), 50000))):
        t = proc_img(row.get('image'))
        if t is None: continue
        lbl = 1 if 'fake' in str(row.get('label','')).lower() else 0
        if lbl == 0: real_frames.append(t)
        else: fake_frames.append(t)
    print(f'  DF-Real: real={len(real_frames):,} fake={len(fake_frames):,}')
except Exception as e: print(f'  Failed: {e}')

# 2. Celeb-DF faces
print('Loading Celeb-DF...')
try:
    cdf = load_dataset('haywhy/celeb-df-v2', split='train', trust_remote_code=True)
    for row in cdf.select(range(min(len(cdf), 30000))):
        t = proc_img(row.get('image'))
        if t is None: continue
        lbl = int(row.get('label', 0))
        if lbl == 0: real_frames.append(t)
        else: fake_frames.append(t)
    print(f'  Celeb-DF: real={len(real_frames):,} fake={len(fake_frames):,}')
except Exception as e: print(f'  Celeb-DF failed: {e}')

# 3. CIFAKE as additional real image baseline
print('Loading CIFAKE real baseline...')
try:
    cf = load_dataset('jungwoo-ha/CIFAKE', split='train', trust_remote_code=True)
    for row in cf.select(range(min(len(cf), 20000))):
        if row['label'] == 0:  # real only
            t = proc_img(row.get('image'))
            if t is not None: real_frames.append(t)
except Exception as e: print(f'  CIFAKE failed: {e}')

print(f'Total frames: real={len(real_frames):,} | fake={len(fake_frames):,}')

In [ ]:
real_s = random.sample(real_frames, min(len(real_frames), N_PER_CLASS))
fake_s = random.sample(fake_frames, min(len(fake_frames), N_PER_CLASS))
all_data = [(t,0) for t in real_s] + [(t,1) for t in fake_s]
random.shuffle(all_data)
split = int(len(all_data)*0.90)
train_data, val_data = all_data[:split], all_data[split:]
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

class FrameDS(torch.utils.data.Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        t, l = self.d[i]
        return {'pixel_values': t, 'labels': torch.tensor(l, dtype=torch.long)}

train_ds = FrameDS(train_data)
val_ds   = FrameDS(val_data)
print('Dataset ready')

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model, TaskType

processor = ViTImageProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model = ViTForImageClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'real', 1:'fake'},
    label2id={'real':0, 'fake':1},
    ignore_mismatched_sizes=True, token=HF_TOKEN,
)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_R*2,
    lora_dropout=0.05, bias='none',
    target_modules=['query','key','value','dense'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def collate(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}

def metrics(ep):
    logits, labels = ep
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds),
            'roc_auc': roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR,
    weight_decay=0.01, warmup_ratio=0.05, lr_scheduler_type='cosine',
    evaluation_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1',
    fp16=torch.cuda.is_available(), logging_steps=100,
    report_to='none', remove_unused_columns=False, seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=metrics, data_collator=collate,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Training...')
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

res = trainer.evaluate()
acc, f1, auc = res.get('eval_accuracy',0), res.get('eval_f1',0), res.get('eval_roc_auc',0)
print(f'Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}%')
print('PASSED' if acc>=0.80 else 'Below target')

po = trainer.predict(val_ds)
preds  = np.argmax(po.predictions, axis=-1)
labels = po.label_ids
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], cmap='Purples')
plt.title(f'Video Frame Acc={acc*100:.1f}%')
plt.tight_layout(); plt.show()
print(classification_report(labels, preds, target_names=['real','fake']))

merged = model.merge_and_unload()
merged.push_to_hub(PUSH_REPO, token=HF_TOKEN)
processor.push_to_hub(PUSH_REPO, token=HF_TOKEN)
print(f'Live at https://huggingface.co/{PUSH_REPO}')
print('Add to hf-analyze.ts: MODELS.video_finetuned = saghi776/aiscern-video-detector')